In [ ]:
import json
import os
from typing import Any

root_dir = "../../../results/cof_matched_rth_0.99_smape_pa/greedy_forward_selection_per_model/"

results = "greedy_forward_selection_results.json"
evaluations = "greedy_forward_selection_evaluations.json"

with open(os.path.join(root_dir, results)) as f:
    results = json.load(f)
with open(os.path.join(root_dir, evaluations)) as f:
    evaluations = json.load(f)

In [ ]:
import pandas as pd

DATASET_LABELS = {
    "ecfp": "ecfp",
    "functional_groups": "functional groups",
    "maccs": "maccs",
    "descriptor": "custom descriptor",
    "estate_sum": "estate",
    "custom_pattern": "custom patterns",
}

DATASET_DISPLAY_NAMES = {
    "ecfp": "ECFP",
    "functional groups": "Functional groups",
    "maccs": "MACCS",
    "estate": "EState",
    "custom descriptor": "Custom descriptor",
    "custom patterns": "Custom patterns",
    "cusotm patterns": "Custom patterns",
}

MODEL_DISPLAY_NAMES = {
    "bayes": "BRR",
    "chemprop": "ChemProp",
    "knn": "KNN",
    "lasso": "LR",
    "mlp": "MLP",
    "random_forest": "RF",
    "tabpfn": "TabPFN",
    "xgboost": "XGB",
}


def normalize_dataset_name(name: str):
    for key, label in DATASET_LABELS.items():
        if key in name:
            return label
    raise ValueError(f"Unknown dataset name: {name}")


def normalize_raw_candidate(raw_candidate: str):
    names: list[str] = sorted(raw_candidate.split("__"))
    return " + ".join(normalize_dataset_name(name) for name in names)


def prettify_dataset_label(label: str) -> str:
    parts = [part.strip() for part in label.split(" + ")]
    return " + ".join(DATASET_DISPLAY_NAMES.get(part, part) for part in parts)


def per_model_selection(iterations: list[dict[str, Any]], model_name: str):
    model_iterations = [i for i in iterations if i["model"] == model_name]

    selected_datasets_per_step = {"step": [], "datasets": [], "datasets_raw": [], "smape": [], "pa": []}
    all_results = {"datasets": [], "datasets_raw": [], "smape": [], "pa": []}
    for i in model_iterations:
        selected_data_raw = [str(name) for name in i["selected_so_far"]]
        selected_data_combined = "__".join(selected_data_raw)
        step = len(selected_data_raw) - 1
        matching_rankings = [j for j in i["ranking"] if j["candidate"] == selected_data_combined]
        if not matching_rankings:
            raise ValueError(f"Candidate '{selected_data_combined}' not found in ranking for model '{model_name}'")
        selected_ranking = matching_rankings[0]
        smape = selected_ranking["smape"]
        pa = selected_ranking["pa"]
        selected_datasets_per_step["step"].append(step)
        selected_data = sorted(selected_data_raw)
        selected_datasets_per_step["datasets_raw"].append("__".join(selected_data))
        selected_datasets_per_step["datasets"].append(" + ".join(normalize_dataset_name(name) for name in selected_data))
        selected_datasets_per_step["smape"].append(smape)
        selected_datasets_per_step["pa"].append(pa)

        for j in i["ranking"]:
            dname = sorted(str(j["candidate"]).split("__"))
            all_results["datasets_raw"].append("__".join(dname))
            all_results["datasets"].append(" + ".join(normalize_dataset_name(name) for name in dname))
            all_results["smape"].append(j["smape"])
            all_results["pa"].append(j["pa"])

    df = pd.DataFrame(selected_datasets_per_step)
    df["smape_rank"] = df["smape"].rank(method="min")
    df["pa_rank"] = df["pa"].rank(method="min", ascending=False)
    df["combined_rank"] = (df["smape_rank"] + df["pa_rank"]) / 2
    df = df.sort_values(by=["combined_rank", "pa_rank", "smape_rank"], ascending=[True, True, True]).reset_index(drop=True)
    best_dataset = df.loc[0, "datasets"]
    best_dataset_raw = df.loc[0, "datasets_raw"]

    df_all = pd.DataFrame(all_results)
    return df, best_dataset, best_dataset_raw, df_all

In [ ]:
models = set(i["model"] for i in results["iterations"])

models_results = {m: per_model_selection(results["iterations"], m) for m in models}

In [ ]:
for model, (df, best_dataset, best_dataset_raw, df_all) in models_results.items():
    print(f"Model: {model}")
    display(df)
    print(f"Best dataset combination: {best_dataset} ({best_dataset_raw})\n")
    print("=" * 50)

In [ ]:
best_datasets = {model: best_dataset for model, (df, best_dataset, best_dataset_raw, df_all) in models_results.items()}
best_datasets_raw = {model: best_dataset_raw for model, (df, best_dataset, best_dataset_raw, df_all) in models_results.items()}
print("Best dataset combinations for each model:")
for model, best_dataset in best_datasets.items():
    print(f"{model}: {best_dataset} ({best_datasets_raw[model]})")

In [ ]:
import numpy as np

results_for_best_datasets = {"model": [], "dataset": [], "dataset_raw": [], "smape": [], "pa": []}

unique_best_datasets_raw = set(best_datasets_raw.values())
for dname_raw in unique_best_datasets_raw:
    print(dname_raw)
    for model, (df, best_dataset, best_dataset_raw, df_all) in models_results.items():
        if dname_raw in df_all["datasets_raw"].to_list():
            matching_rows = df_all[df_all["datasets_raw"] == dname_raw]
            dataset_label = matching_rows.datasets.values[0]
            smape_result = matching_rows.smape.values[0]
            pa_result = matching_rows.pa.values[0]
        else:
            dataset_label = normalize_raw_candidate(dname_raw)
            smape_result = np.nan
            pa_result = np.nan
        results_for_best_datasets["model"].append(model)
        results_for_best_datasets["dataset"].append(dataset_label)
        results_for_best_datasets["dataset_raw"].append(dname_raw)
        results_for_best_datasets["smape"].append(smape_result)
        results_for_best_datasets["pa"].append(pa_result)

df_results = pd.DataFrame(results_for_best_datasets)
df_results

In [ ]:
# Build missing evaluations as (model, non-normalized dataset list) requests.
evaluated_keys = {(record["model"], "__".join(sorted(record["candidate"]))) for record in evaluations}
missing_df = df_results[df_results["smape"].isna()].copy().reset_index(drop=True)
missing_df["datasets"] = missing_df["dataset_raw"].str.split("__")

missing_requests = [
    {
        "model": row["model"],
        "datasets": row["datasets"],
    }
    for _, row in missing_df.iterrows()
    if (row["model"], "__".join(sorted(row["datasets"]))) not in evaluated_keys
]

missing_requests_path = os.path.join(root_dir, "missing_evaluation_requests.json")
with open(missing_requests_path, "w") as f:
    json.dump(missing_requests, f, indent=2)

print(f"Missing requests saved to: {missing_requests_path}")
print(f"Number of missing requests: {len(missing_requests)}")
missing_df

In [ ]:
import itertools
import pickle

raw_datasets = df_results["dataset_raw"].unique()
models = df_results["model"].unique()

results = {
    "model": [],
    "dataset": [],
    "dataset_raw": [],
    "smape_mean": [],
    "smape_std": [],
    "pa_mean": [],
    "pa_std": [],
    "rmse_mean": [],
    "rmse_std": [],
}

for m in models:
    for dname in raw_datasets:
        dname_list = dname.split("__")
        all_permutations = itertools.permutations(dname_list)
        for p in all_permutations:
            pname = "__".join(p)
            results_path = os.path.join(root_dir, "greedy_candidates", m, pname, "scores.pickle")
            try:
                with open(results_path, "rb") as f:
                    r_md = pickle.load(f)
                break
            except FileNotFoundError:
                continue
        mean_pa = np.mean(r_md["pairwise_accuracy_score"])
        mean_smape = np.mean(r_md["smape"])
        mean_rmse = np.mean(r_md["rmse"])
        std_pa = np.std(r_md["pairwise_accuracy_score"])
        std_smape = np.std(r_md["smape"])
        std_rmse = np.std(r_md["rmse"])
        results["model"].append(m)
        results["dataset_raw"].append(dname)
        results["dataset"].append(" + ".join(normalize_dataset_name(name) for name in dname.split("__")))
        results["smape_mean"].append(mean_smape)
        results["smape_std"].append(std_smape)
        results["pa_mean"].append(mean_pa)
        results["pa_std"].append(std_pa)
        results["rmse_mean"].append(mean_rmse)
        results["rmse_std"].append(std_rmse)
df_final_results = pd.DataFrame(results)
df_final_results

In [ ]:
smape_mean_matrix = df_final_results.pivot(index="dataset", columns="model", values="smape_mean")
pa_mean_matrix = df_final_results.pivot(index="dataset", columns="model", values="pa_mean")
rmse_mean_matrix = df_final_results.pivot(index="dataset", columns="model", values="rmse_mean")

smape_std_matrix = df_final_results.pivot(index="dataset", columns="model", values="smape_std")
pa_std_matrix = df_final_results.pivot(index="dataset", columns="model", values="pa_std")
rmse_std_matrix = df_final_results.pivot(index="dataset", columns="model", values="rmse_std")

smape_matrix = (smape_mean_matrix * 100).round(1).astype(str) + "(" + smape_std_matrix.round(2).astype(str) + ")"
pa_matrix = (pa_mean_matrix * 100).round(1).astype(str) + "(" + pa_std_matrix.round(2).astype(str) + ")"
rmse_matrix = rmse_mean_matrix.round(1).astype(str) + "(" + rmse_std_matrix.round(1).astype(str) + ")"

In [ ]:
smape_mean_matrix

In [ ]:
pa_mean_matrix

In [ ]:
def prettify_df(df):
    df.index = [prettify_dataset_label(str(label)) for label in df.index]
    df.columns = [MODEL_DISPLAY_NAMES.get(str(column), str(column)) for column in df.columns]
    return df


smape_matrix = prettify_df(smape_matrix)
pa_matrix = prettify_df(pa_matrix)
rmse_matrix = prettify_df(rmse_matrix)
smape_mean_matrix = prettify_df(smape_mean_matrix)
pa_mean_matrix = prettify_df(pa_mean_matrix)
rmse_mean_matrix = prettify_df(rmse_mean_matrix)


smape_ranking = smape_mean_matrix.stack(dropna=False).rank(method="min", ascending=True).unstack()
pa_ranking = pa_mean_matrix.stack(dropna=False).rank(method="min", ascending=False).unstack()

combined_ranking = (pa_ranking + smape_ranking) / 2

In [ ]:
import matplotlib as mpl
import seaborn as sns
from matplotlib import pyplot as plt

FONT_HEATMAP = 11
FONT_AX = 11
FONT_TITLE = 12


def plot_smape_pa(smape, pa, smape_annot, pa_annot, f_name=""):
    fig, axes = plt.subplots(2, 1, figsize=(14, 10))
    sns.heatmap(smape, annot=smape_annot, fmt="", cmap="magma_r", cbar=False, ax=axes[0], annot_kws={"size": FONT_HEATMAP})
    axes[0].set_title("SMAPE (%)", fontsize=FONT_TITLE)
    sns.heatmap(pa, annot=pa_annot, fmt="", cmap="magma", cbar=False, ax=axes[1], annot_kws={"size": FONT_HEATMAP})
    axes[1].set_title("PA (%)", fontsize=FONT_TITLE)

    for ax in axes:
        ax.tick_params(axis="x", labelsize=FONT_AX)
        ax.tick_params(axis="y", labelsize=FONT_AX)

    plt.tight_layout(rect=[0, 0, 0.98, 1])
    cbar_ax = fig.add_axes([0.99, 0.25, 0.01, 0.5])

    cmap = mpl.cm.magma
    norm = mpl.colors.Normalize(vmin=0, vmax=1)  # Dummy normalization

    cb = mpl.colorbar.ColorbarBase(cbar_ax, cmap=cmap, norm=norm, orientation="vertical")
    cb.set_ticks([0, 1])
    cb.set_ticklabels(["Worst", "Best"])

    if f_name:
        plt.savefig(f"{f_name}.png", dpi=500, pad_inches=0, bbox_inches="tight")

    plt.show()


def plot_rmse(rmse, rmse_annot, f_name=""):
    fig, axes = plt.subplots(1, 1, figsize=(14, 5))
    sns.heatmap(rmse, annot=rmse_annot, fmt="", cmap="magma_r", cbar=False, ax=axes, annot_kws={"size": FONT_HEATMAP})
    axes.set_title("RMSE (F/g)", fontsize=FONT_TITLE)

    axes.tick_params(axis="x", labelsize=FONT_AX)
    axes.tick_params(axis="y", labelsize=FONT_AX)

    plt.tight_layout(rect=[0, 0, 0.98, 1])
    cbar_ax = fig.add_axes([0.99, 0.25, 0.01, 0.5])

    cmap = mpl.cm.magma
    norm = mpl.colors.Normalize(vmin=0, vmax=1)  # Dummy normalization

    cb = mpl.colorbar.ColorbarBase(cbar_ax, cmap=cmap, norm=norm, orientation="vertical")
    cb.set_ticks([0, 1])
    cb.set_ticklabels(["Worst", "Best"])

    if f_name:
        plt.savefig(f"{f_name}.png", dpi=500, pad_inches=0, bbox_inches="tight")

    plt.show()


def plot_ranking(rankings, f_name=""):
    fig, axes = plt.subplots(len(rankings.keys()), 1, figsize=(14, 5 * len(rankings.keys())))
    if len(rankings.keys()) == 1:
        rname, ranking = list(rankings.keys())[0], list(rankings.values())[0]
        sns.heatmap(ranking, annot=True, fmt=".2f", cmap="magma_r", cbar=False, ax=axes, annot_kws={"size": FONT_HEATMAP})
        axes.set_title(rname, fontsize=FONT_TITLE)
        axes.tick_params(axis="x", labelsize=FONT_AX)
        axes.tick_params(axis="y", labelsize=FONT_AX)
    else:
        for i, (rname, ranking) in enumerate(rankings.items()):
            sns.heatmap(ranking, annot=True, fmt=".2f", cmap="magma_r", cbar=False, ax=axes[i], annot_kws={"size": FONT_HEATMAP})
            axes[i].set_title(rname, fontsize=FONT_TITLE)
            axes[i].tick_params(axis="x", labelsize=FONT_AX)
            axes[i].tick_params(axis="y", labelsize=FONT_AX)
    plt.tight_layout(rect=[0, 0, 0.98, 1])
    cbar_ax = fig.add_axes([0.99, 0.25, 0.01, 0.5])

    cmap = mpl.cm.magma
    norm = mpl.colors.Normalize(vmin=0, vmax=1)  # Dummy normalization

    cb = mpl.colorbar.ColorbarBase(cbar_ax, cmap=cmap, norm=norm, orientation="vertical")
    cb.set_ticks([0, 1])
    cb.set_ticklabels(["Worst", "Best"])

    if f_name:
        plt.savefig(f"{f_name}.png", dpi=500, pad_inches=0, bbox_inches="tight")

    plt.show()


plot_smape_pa(smape_mean_matrix, pa_mean_matrix, smape_matrix, pa_matrix, f_name="smape_pa_heatmap")
plot_rmse(rmse_mean_matrix, rmse_matrix, f_name="rmse_heatmap")
plot_ranking({"Combined Rank": combined_ranking}, f_name="combined_ranking_heatmap")
plot_ranking({"SMAPE Rank": smape_ranking, "PA Rank": pa_ranking}, f_name="smape_pa_ranking_heatmap")

## XAI analysis of best models

Best models are identified as:
- BRR: Custom descriptor + ECFP + EState + Functional groups
- RF: EState + Functional groups

In [ ]:
best_datasets

In [ ]:
import joblib

bayes_best_path = os.path.join(
    root_dir,
    "greedy_candidates",
    "bayes",
    "features_rth_0.99_ecfp_2_1024_True__features_rth_0.99_functional_groups_True__features_rth_0.99_descriptor_False_False__features_rth_0.99_estate_sum",
    "feature_importance.pickle",
)
rf_best_path = os.path.join(root_dir, "greedy_candidates", "random_forest", "features_rth_0.99_estate_sum__features_rth_0.99_functional_groups_True", "feature_importance.pickle")

with open(bayes_best_path, "rb") as f:
    bayes_shap = joblib.load(f)
with open(rf_best_path, "rb") as f:
    rf_shap = joblib.load(f)

brr_imp_whole = "brr_best_model/feature_importance.pickle"
rf_imp_whole = "rf_best_model/feature_importance.pickle"

with open(brr_imp_whole, "rb") as f:
    brr_whole_imp = joblib.load(brr_imp_whole)

with open(rf_imp_whole, "rb") as f:
    rf_whole_imp = joblib.load(rf_imp_whole)

brr_imp_whole_shap = "brr_best_model/feature_importance_shap.pickle"
with open(brr_imp_whole_shap, "rb") as f:
    brr_imp_whole_shap = joblib.load(brr_imp_whole_shap)

In [ ]:
def process_bayes(bayes_shap):
    features = set()
    for _, exp in bayes_shap:
        features.update(exp["feature"])
    features = list(features)
    importances = {f: [] for f in features}
    for _, exp in bayes_shap:
        dict_exp = dict(zip(exp["feature"], exp["coefficient"]))
        for f in importances.keys():
            imp = np.array(dict_exp[f])
            importances[f].append(imp)
    importances_mean = {f: np.mean(np.abs(vals)) for f, vals in importances.items()}
    importances_df = pd.DataFrame.from_dict(importances_mean, orient="index", columns=["abs_importance"])
    importances_df["all_importances"] = [i for _, i in importances.items()]
    importances_df.index.name = "Feature"
    importances_df.reset_index(inplace=True, drop=False)
    importances_df.sort_values(by=["abs_importance"], ascending=False, inplace=True)
    importances_df = importances_df[importances_df["Feature"] != "intercept"]
    return importances_df


bayes_importances_df = process_bayes(bayes_shap)
bayes_importances_df

In [ ]:
from typing import Literal, Sequence

import matplotlib.cm as cm
import matplotlib.colors as mcolors


def _get_red_blue_cmap() -> mcolors.LinearSegmentedColormap:
    """Creates a red-blue colormap with a smooth transition from blue to red.

    Returns:
        A colormap object that transitions from blue to red.
    """
    gray_rgb = np.array([0.51615537, 0.51615111, 0.5161729])

    cdict: dict[Literal["red", "green", "blue", "alpha"], Sequence[tuple[float, float, float]]] = {
        "red": [
            (0.0, 0.0, 0.0),
            (0.494949494949495, 0.6035590338007161, 0.6035590338007161),
            (1.0, 1.0, 1.0),
        ],
        "green": [
            (0.0, 0.5433775692459107, 0.5433775692459107),
            (0.494949494949495, 0.14541587318267168, 0.14541587318267168),
            (1.0, 0.0, 0.0),
        ],
        "blue": [
            (0.0, 0.983379062301401, 0.983379062301401),
            (0.494949494949495, 0.6828490076357064, 0.6828490076357064),
            (1.0, 0.31796406298163893, 0.31796406298163893),
        ],
        "alpha": [(0, 1.0, 1.0), (0.494949494949495, 1.0, 1.0), (1.0, 1.0, 1.0)],
    }
    red_blue = mcolors.LinearSegmentedColormap("red_blue", cdict)
    red_blue.set_bad(gray_rgb.tolist(), 1.0)
    red_blue.set_over(gray_rgb.tolist(), 1.0)
    red_blue.set_under(gray_rgb.tolist(), 1.0)
    return red_blue


def _format_feature_name(feature_name: str) -> str:
    functional_group_mapping = [
        "aliphatic carboxylic acids",
        "aliphatic hydroxyl groups",
        "aliphatic hydroxyl groups excluding tert-OH",
        "N functional groups attached to aromatics",
        "Aromatic carboxylic acide",
        "aromatic nitrogens",
        "aromatic amines",
        "aromatic hydroxyl groups",
        "carboxylic acids COO",
        "carboxylic acids COO2",
        "carbonyl O",
        "carbonyl O, excluding COOH",
        "thiocarbonyl",
        "C(OH)CCN-Ctert-alkyl or  C(OH)CCNcyclic",
        "Imines",
        "Tertiary amines",
        "Secondary amines",
        "Primary amines",
        "hydroxylamine groups",
        "XCCNR groups",
        "tert-alicyclic amines (no heteroatoms, not quinine-like bridged N)",
        "H-pyrrole nitrogens",
        "thiol groups",
        "aldehydes",
        "alkyl carbamates (subject to hydrolysis)",
        "alkyl halides",
        "Allylic oxidation sites excluding steroid dienone",
        "amides",
        "amidine groups",
        "anilines",
        "aryl methyl sites for hydroxylation",
        "azide groups",
        "azo groups",
        "barbiturate groups",
        "Benzene rings",
        "benzodiazepines with no additional fused rings",
        "Bicyclic",
        "diazo groups",
        "dihydropyridines",
        "epoxide rings",
        "esters",
        "ether oxygens (including phenoxy)",
        "furan rings",
        "guanidine groups",
        "halogens",
        "hydrazine groups",
        "hydrazone groups",
        "imidazole rings",
        "imide groups",
        "isocyanates",
        "isothiocyanates",
        "ketones",
        "ketones excluding diaryl, a,b-unsat. dienones, heteroatom on Calpha",
        "beta lactams",
        "cyclic esters (lactones)",
        "methoxy groups -OCH3",
        "morpholine rings",
        "nitriles",
        "nitro groups",
        "nitro benzene ring substituents",
        "non-ortho nitro benzene ring substituents",
        "nitroso groups, excluding NO2",
        "oxazole rings",
        "oxime groups",
        "para-hydroxylation sites",
        "phenols",
        "phenolic OH excluding ortho intramolecular Hbond substituents",
        "phosphoric acid groups",
        "phosphoric ester groups",
        "piperdine rings",
        "piperzine rings",
        "primary amides",
        "primary sulfonamides",
        "pyridine rings",
        "quaternary nitrogens",
        "thioether",
        "sulfonamides",
        "sulfone groups",
        "terminal acetylenes",
        "tetrazole rings",
        "thiazole rings",
        "thiocyanates",
        "thiophene rings",
        "unbranched alkanes of at least 4 members (excludes halogenated alkanes)",
        "urea groups",
    ]
    functional_group_mapping = {f"functional_groups_{i}": name for i, name in enumerate(functional_group_mapping)}
    if feature_name.startswith("estate_"):
        suffix = feature_name.removeprefix("estate_")
        return f"EState {suffix}"

    if feature_name.startswith("ecfp_"):
        suffix = feature_name.removeprefix("ecfp_")
        return f"ECFP {suffix}"

    if feature_name.startswith("functional_groups_"):
        suffix = feature_name.removeprefix("functional_groups_")
        if functional_group_mapping is None:
            return f"Functional group {suffix}"

        mapped_name = functional_group_mapping.get(suffix) or functional_group_mapping.get(feature_name) or functional_group_mapping.get(int(suffix)) if suffix.isdigit() else None
        return mapped_name if mapped_name else f"Functional group {suffix}"

    return feature_name


def _format_feature_names(
    feature_names: list[str],
) -> list[str]:
    return [_format_feature_name(name) for name in feature_names]


def shap_correlations(shap_explanation, data_columns, k=10):
    data_values = pd.DataFrame(shap_explanation.data, columns=data_columns)
    shap_values = pd.DataFrame(data=shap_explanation.values, columns=data_columns)
    absolute_means = shap_values.abs().mean(axis=0)
    absolute_means = absolute_means.reset_index(drop=False)
    absolute_means.rename(columns={"index": "feature", 0: "value"}, inplace=True)
    absolute_means = absolute_means.sort_values(by="value").tail(k)
    correlations = []
    for fname in absolute_means["feature"]:
        fvalues = data_values[fname]
        svalues = shap_values[fname]
        corr = fvalues.corr(svalues)
        correlations.append(corr)
    absolute_means["corr"] = correlations

    values = absolute_means["value"]
    feature_names = _format_feature_names(
        absolute_means["feature"].tolist(),
    )

    # 3. Plotting
    cmap = plt.get_cmap("RdBu")
    norm = mcolors.Normalize(vmin=-1.0, vmax=1.0)
    bar_colors = [cmap(norm(c)) for c in absolute_means["corr"]]

    fig, ax = plt.subplots(figsize=(9, len(values) * 0.6))
    y_pos = np.arange(len(values))

    bars = ax.barh(y_pos, values, color=bar_colors, height=0.7)

    ax.xaxis.set_ticks_position("bottom")
    ax.yaxis.set_ticks_position("none")
    ax.spines["right"].set_visible(False)
    ax.spines["top"].set_visible(False)

    ax.set_yticks(y_pos)
    ax.set_yticklabels(feature_names, fontsize=13)
    ax.set_xlabel("mean(|SHAP value|)", fontsize=13)

    # Add horizontal dotted lines behind bars
    for i in y_pos:
        ax.axhline(i, color="#888888", lw=0.5, dashes=(1, 5), zorder=-1)

    # 5. Add text labels at the end of bars
    max_val = np.max(values)
    for bar, val in zip(bars, values):
        ax.text(bar.get_width() + (max_val * 0.01), bar.get_y() + bar.get_height() / 2, f"{val:.2f}", va="center", ha="left", fontsize=12, color="black")

    sm = cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    cbar = plt.colorbar(sm, ax=ax, fraction=0.046, pad=0.04, aspect=50, ticks=[-1, -0.5, 0, 0.5, 1])
    cbar.set_label("Feature vs SHAP Correlation", fontsize=11)
    cbar.outline.set_visible(False)

    # Ensure there is room for the labels
    ax.set_xlim(0, max_val * 1.2)
    return ax

In [ ]:
def process_rf_whole(
    rf_shap,
    selected_feature_sets: list[str] | str,
    datasets_dir: str = "../../../data/predictor/cof_matched/rth_0.99",
):
    import pandas as pd
    import shap

    def resolve_feature_set_name(feature_set_name: str, available_feature_files: set[str]) -> str:
        stem = feature_set_name.replace(".csv", "")
        if stem not in available_feature_files:
            raise ValueError(f"Feature set '{feature_set_name}' was not found in '{datasets_dir}/features'.")
        return stem

    def build_combined_dataset(
        feature_sets: list[str],
        target_column: str = "capacity_max",
        id_column: str = "smiles",
    ) -> pd.DataFrame:
        if len(feature_sets) == 0:
            raise ValueError("selected_feature_sets cannot be empty.")

        feature_dir = os.path.join(datasets_dir, "features")
        available_files = {file_name.replace(".csv", "") for file_name in os.listdir(feature_dir)}
        resolved_names = [resolve_feature_set_name(name, available_files) for name in feature_sets]
        loaded_sets = {name: pd.read_csv(os.path.join(feature_dir, f"{name}.csv")) for name in resolved_names}

        first_df = loaded_sets[resolved_names[0]]
        keep_columns = [target_column]
        if id_column in first_df.columns:
            keep_columns.insert(0, id_column)
        combined = first_df[keep_columns].copy()

        selected_columns: list[str] = []
        for feature_set_name in resolved_names:
            dataset_part = loaded_sets[feature_set_name]
            candidate_columns = [c for c in dataset_part.columns if c not in {target_column, id_column}]
            new_columns = [c for c in candidate_columns if c not in selected_columns]

            if len(new_columns) == 0:
                continue

            if id_column in combined.columns and id_column in dataset_part.columns:
                aligned = dataset_part.set_index(id_column).reindex(combined[id_column])
                combined[new_columns] = aligned[new_columns].to_numpy()
            else:
                if len(dataset_part) != len(combined):
                    raise ValueError(f"Cannot align feature set '{feature_set_name}' without id column '{id_column}'.")
                combined[new_columns] = dataset_part.reset_index(drop=True)[new_columns]

            selected_columns.extend(new_columns)

        return combined

    if isinstance(selected_feature_sets, str):
        selected_feature_sets = [name for name in selected_feature_sets.split("__") if name]

    dataset = build_combined_dataset(selected_feature_sets)
    X = dataset.drop(columns=["capacity_max", "smiles"])

    X_columns = X.columns.tolist()
    X_values = X.values

    f_imp_all = rf_shap[1]
    explanation = shap.Explanation(values=f_imp_all, data=X_values, feature_names=X_columns)
    return explanation


rf_explanation_whole = process_rf_whole(rf_whole_imp, ["features_rth_0.99_estate_sum", "features_rth_0.99_functional_groups_True"])

In [ ]:
ax = shap_correlations(rf_explanation_whole, rf_explanation_whole.feature_names, k=10)
plt.tight_layout()
# plt.savefig('images/shap_importance_rf_whole.png', dpi=500)

In [ ]:
brr_explanation_whole_shap = process_rf_whole(
    brr_imp_whole_shap, "features_rth_0.99_ecfp_2_1024_True__features_rth_0.99_functional_groups_True__features_rth_0.99_descriptor_False_False__features_rth_0.99_estate_sum".split("__")
)
ax = shap_correlations(brr_explanation_whole_shap, brr_explanation_whole_shap.feature_names, k=10)
plt.tight_layout()
# plt.savefig('images/shap_importance_brr_whole.png', dpi=500)

In [ ]:
def visualize_bayes(bayes_importances_df, k=10):
    plot_df = bayes_importances_df.sort_values(by=["abs_importance"], ascending=False).head(k).copy()

    # Use the per-fold coefficients to infer the signed effect direction.
    plot_df["signed_mean"] = plot_df["all_importances"].apply(lambda vals: float(np.mean(vals)))
    plot_df["sign_consistency"] = plot_df["all_importances"].apply(lambda vals: float(np.mean(np.sign(vals) == np.sign(np.mean(vals)))) if len(vals) else np.nan)

    # Reverse order so the largest-magnitude feature appears at the bottom.
    plot_df = plot_df.iloc[::-1].reset_index(drop=True)

    values = plot_df["abs_importance"].to_numpy()
    signed_mean = plot_df["signed_mean"].to_numpy()
    sign_consistency = plot_df["sign_consistency"].to_numpy()
    feature_names = _format_feature_names(
        plot_df["Feature"].tolist(),
    )
    y_pos = np.arange(len(values))

    # Neutral color for features with almost-zero mean effect.
    eps = 1e-6
    bar_colors = ["#d62728" if v < -eps else "#1f77b4" if v > eps else "#7f7f7f" for v in signed_mean]

    fig, ax = plt.subplots(figsize=(12, len(values) * 0.6))
    bars = ax.barh(y_pos, values, color=bar_colors, height=0.7)

    ax.xaxis.set_ticks_position("bottom")
    ax.yaxis.set_ticks_position("none")
    ax.spines["right"].set_visible(False)
    ax.spines["top"].set_visible(False)

    ax.set_yticks(y_pos)
    ax.set_yticklabels(feature_names, fontsize=13)
    ax.set_xlabel("|Coefficient value|", fontsize=13)

    for i in y_pos:
        ax.axhline(i, color="#888888", lw=0.5, dashes=(1, 5), zorder=-1)

    max_abs = float(np.max(values)) if len(values) else 0.0
    offset = max(max_abs * 0.01, 1e-6)
    for bar, mu, consistency in zip(bars, values, sign_consistency):
        ax.text(bar.get_width() + offset, bar.get_y() + bar.get_height() / 2, f"{mu:.3f}", va="center", ha="left", fontsize=12, color="black")

    from matplotlib.patches import Patch

    legend_handles = [
        Patch(facecolor="#1f77b4", edgecolor="none", label="Positive"),
        Patch(facecolor="#d62728", edgecolor="none", label="Negative"),
    ]
    ax.legend(handles=legend_handles, title="Coefficient sign", frameon=False, loc="center left", bbox_to_anchor=(1.02, 0.5), borderaxespad=0.0)

    # Reserve room on the right for the external legend.
    fig.subplots_adjust(right=0.78)

    ax.set_xlim(0, max_abs * 1.25 if max_abs > 0 else 1)
    return ax


ax = visualize_bayes(bayes_importances_df)
plt.tight_layout()
# plt.savefig('importance_brr.png', dpi=500)